In [4]:
import pandas as pd
df = pd.read_csv('tourism_data.csv')

In [5]:
df.head()

,booking_id,tourist_country,hotel_city,arrival_date,hotel_stars,meal_plan,room_type,nights,price_per_night,transport_type,booking_channel
0,B0001,UK,Kandy,2024-10-14,3,Full Board,Standard,4,8142.0,Flight,Agent
1,B0002,UK,Negombo,2024-06-21,3,Room Only,Suite,1,7774.0,Bus,Direct
2,B0003,Germany,Negombo,2024-07-12,3,Half Board,Deluxe,5,7507.0,Flight,Online
3,B0004,Russia,Negombo,2024-02-21,4,Half Board,Suite,6,16002.0,Train,Direct
4,B0005,USA,Galle,2024-02-06,5,Breakfast,Standard,2,24811.0,Bus,Agent


In [7]:
df.dtypes

booking_id             str
tourist_country        str
hotel_city             str
arrival_date           str
hotel_stars          int64
meal_plan              str
room_type              str
nights               int64
price_per_night    float64
transport_type         str
booking_channel        str
dtype: object

In [8]:
df.describe

<bound method NDFrame.describe of      booking_id tourist_country hotel_city arrival_date  hotel_stars  \
0         B0001              UK      Kandy   2024-10-14            3   
1         B0002              UK    Negombo   2024-06-21            3   
2         B0003         Germany    Negombo   2024-07-12            3   
3         B0004          Russia    Negombo   2024-02-21            4   
4         B0005             USA      Galle   2024-02-06            5   
...         ...             ...        ...          ...          ...   
1995      B1996             USA      Kandy   2024-10-07            4   
1996      B1997          France      Kandy   2024-12-24            3   
1997      B1998         Germany    Negombo   2024-02-16            4   
1998      B1999           India      Kandy   2024-01-12            4   
1999      B2000           China    Negombo   2024-01-14            3   

       meal_plan room_type  nights  price_per_night transport_type  \
0     Full Board  Standard     

In [9]:
issues_found = []
df_clean=df.copy()

In [10]:
#check null values
missing = df_clean.isnull().sum()
if missing.sum() > 0:
    issues_found.append(f"Found {missing.sum()} missing values")
    print(f"Missing values:\n{missing[missing > 0]}")
else:
    print("No missing values")

Missing values:
price_per_night    12
dtype: int64


In [11]:
#Check duplicates
duplicates = df_clean.duplicated(subset=['booking_id']).sum()
if duplicates > 0:
    issues_found.append(f"Found {duplicates} duplicates")
    df_clean = df_clean.drop_duplicates(subset=['booking_id'])
    print(f"Removed {duplicates} duplicates")
else:
    print("No duplicates")

No duplicates


In [12]:
#check invalid
invalid = (df_clean['price_per_night'] <= 0).sum()

issues_found.append(f"Found {invalid} invalid prices")

if invalid > 0:
    print(f"Invalid records found: {invalid}")
    df_clean = df_clean[df_clean['price_per_night'] > 0]
else:
    print("No invalid records")

Invalid records found: 9


In [14]:
if (df_clean['price_per_night'] <= 0).sum() > 0:
    invalid = (df_clean['price_per_night'] <= 0).sum()
    issues_found.append(f"Found {invalid} invalid prices")
    df_clean = df_clean[df_clean['price_per_night'] > 0]
    print(f"Removed {invalid} invalid prices")


In [16]:
print(f"\nAfter cleaning: {len(df_clean)} records")
print(f"Issues found: {len(issues_found)}")
for issue in issues_found:
    print(f"   - {issue}")
    


After cleaning: 1979 records
Issues found: 2
   - Found 12 missing values
   - 9


In [17]:
# Extract date features
df_clean['arrival_date'] = pd.to_datetime(df_clean['arrival_date'])
df_clean['month'] = df_clean['arrival_date'].dt.month
df_clean['day_of_week'] = df_clean['arrival_date'].dt.day_name()
df_clean['quarter'] = df_clean['arrival_date'].dt.quarter

In [18]:
# create total cost
df_clean['total_cost'] = df_clean['nights'] * df_clean['price_per_night']
print("Created: total_cost")

Created: total_cost


In [19]:
# Create spend 
df_clean['spend_category'] = pd.cut(df_clean['total_cost'],
                                      bins=[0, 20000, 50000, 200000],
                                      labels=['Low Spender', 'Medium Spender', 'High Spender'])
print("Created: spend_category")

Created: spend_category


In [20]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 1979 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   booking_id       1979 non-null   str           
 1   tourist_country  1979 non-null   str           
 2   hotel_city       1979 non-null   str           
 3   arrival_date     1979 non-null   datetime64[us]
 4   hotel_stars      1979 non-null   int64         
 5   meal_plan        1979 non-null   str           
 6   room_type        1979 non-null   str           
 7   nights           1979 non-null   int64         
 8   price_per_night  1979 non-null   float64       
 9   transport_type   1979 non-null   str           
 10  booking_channel  1979 non-null   str           
 11  month            1979 non-null   int32         
 12  day_of_week      1979 non-null   str           
 13  quarter          1979 non-null   int32         
 14  total_cost       1979 non-null   float64       
 15  spe

In [21]:
df_clean.head(10)

,booking_id,tourist_country,hotel_city,arrival_date,hotel_stars,meal_plan,room_type,nights,price_per_night,transport_type,booking_channel,month,day_of_week,quarter,total_cost,spend_category
0,B0001,UK,Kandy,2024-10-14,3,Full Board,Standard,4,8142.0,Flight,Agent,10,Monday,4,32568.0,Medium Spender
1,B0002,UK,Negombo,2024-06-21,3,Room Only,Suite,1,7774.0,Bus,Direct,6,Friday,2,7774.0,Low Spender
2,B0003,Germany,Negombo,2024-07-12,3,Half Board,Deluxe,5,7507.0,Flight,Online,7,Friday,3,37535.0,Medium Spender
3,B0004,Russia,Negombo,2024-02-21,4,Half Board,Suite,6,16002.0,Train,Direct,2,Wednesday,1,96012.0,High Spender
4,B0005,USA,Galle,2024-02-06,5,Breakfast,Standard,2,24811.0,Bus,Agent,2,Tuesday,1,49622.0,Medium Spender
5,B0006,USA,Colombo,2024-01-17,4,Full Board,Deluxe,1,14132.0,Bus,Agent,1,Wednesday,1,14132.0,Low Spender
6,B0007,Australia,Kandy,2024-05-15,3,Breakfast,Deluxe,6,8135.0,Taxi,Direct,5,Wednesday,2,48810.0,Medium Spender
7,B0008,Germany,Colombo,2024-01-25,3,Breakfast,Standard,7,8288.0,Flight,Travel Agency,1,Thursday,1,58016.0,High Spender
8,B0009,Germany,Negombo,2024-01-06,5,Room Only,Deluxe,7,25704.0,Flight,Direct,1,Saturday,1,179928.0,High Spender
9,B0010,Russia,Negombo,2024-05-14,5,Breakfast,Budget,7,25626.0,Train,Agent,5,Tuesday,2,179382.0,High Spender


In [22]:
# Save to CSV
df_clean.to_csv('cleaned_tourism_data.csv', index=False)
print("Saved to 'cleaned_tourism_data.csv'")

Saved to 'cleaned_tourism_data.csv'


In [23]:
import sqlite3

# Create SQLite connection 
conn = sqlite3.connect('tourism_database.db')
print("Connected to SQLite database")

Connected to SQLite database


In [26]:
# Load DataFrame to SQLite
table_name = "SL_tourism_bookings"
df_clean.to_sql(table_name, conn, if_exists="replace", index=False)
print(f"Loaded {len(df_clean)} records to table '{table_name}'")

Loaded 1979 records to table 'SL_tourism_bookings'


In [27]:
df = pd.read_sql("SELECT * FROM SL_tourism_bookings LIMIT 10;", conn)

print(df)

  booking_id tourist_country hotel_city         arrival_date  hotel_stars  \
0      B0001              UK      Kandy  2024-10-14 00:00:00            3   
1      B0002              UK    Negombo  2024-06-21 00:00:00            3   
2      B0003         Germany    Negombo  2024-07-12 00:00:00            3   
3      B0004          Russia    Negombo  2024-02-21 00:00:00            4   
4      B0005             USA      Galle  2024-02-06 00:00:00            5   
5      B0006             USA    Colombo  2024-01-17 00:00:00            4   
6      B0007       Australia      Kandy  2024-05-15 00:00:00            3   
7      B0008         Germany    Colombo  2024-01-25 00:00:00            3   
8      B0009         Germany    Negombo  2024-01-06 00:00:00            5   
9      B0010          Russia    Negombo  2024-05-14 00:00:00            5   

    meal_plan room_type  nights  price_per_night transport_type  \
0  Full Board  Standard       4           8142.0         Flight   
1   Room Only     